# 04. 최종 모델을 작은 GPU에서 재기 (T4 또는 L4, 약 30분 / 1유닛 이하)
03에서 드라이브에 저장한 최종 모델(배포 형식 Q4_K_M)의 속도·전력·메모리를 잽니다.
**T4 런타임에서 한 번, L4 런타임에서 한 번** 실행하면 두 GPU 결과가 모두 남습니다.

In [ ]:
# 저장소 받기 + 패키지 설치
BRANCH = "ccr-a463bea6-bpbyrh"   # main에 합친 뒤에는 "main"
REPO = "github.com/b1bibig/Cfinetune"
import os
try:  # 저장소가 비공개면 코랩 왼쪽 열쇠(보안 비밀)에 GH_TOKEN 을 넣어 두세요
    from google.colab import userdata
    _tok = userdata.get("GH_TOKEN")
    REPO_URL = f"https://{_tok}@{REPO}"
except Exception:
    REPO_URL = f"https://{REPO}"
if not os.path.exists("/content/Cfinetune"):
    !git clone -q -b {BRANCH} {REPO_URL} /content/Cfinetune
%cd /content/Cfinetune
!git pull -q
%pip install -q -r requirements-colab.txt
%pip uninstall -y -q torchao   # 코랩의 옛 torchao가 있으면 PEFT가 LoRA 합치기·불러오기에서 멈춤 (이 실험은 안 씀)

In [ ]:
# 구글 드라이브: 결과(results/)를 세션이 끊겨도 남도록 저장
from google.colab import drive
drive.mount("/content/drive")
import os, shutil, sys
DRIVE = "/content/drive/MyDrive/cfinetune"
os.makedirs(DRIVE, exist_ok=True)

def pull_results():
    if os.path.exists(f"{DRIVE}/results"):
        shutil.copytree(f"{DRIVE}/results", "results", dirs_exist_ok=True)

def push_results():
    shutil.copytree("results", f"{DRIVE}/results", dirs_exist_ok=True)
    print("드라이브에 결과 저장함")

pull_results()
sys.path.insert(0, "scripts"); sys.path.insert(0, "src")

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!bash scripts/build_llama_cpp.sh /content/llama.cpp {DRIVE}/cache

In [ ]:
from _common import load_config
cfg = load_config()
NAME = f"{cfg['lora']['base_variant']}{'-vx' if cfg['vocab']['enabled'] else ''}-lora-{cfg['deploy_quant']}"
!cp {DRIVE}/gguf/{NAME}.gguf /content/
!python scripts/06_bench.py --gguf /content/{NAME}.gguf --name {NAME}
push_results()

## 런타임 끄기

In [ ]:
# 다 끝나면 런타임을 자동으로 끈다 (켜 둔 채 두면 아무것도 안 해도 유닛이 나감). 결과는 이미 드라이브에 있음
# 끄지 않고 더 볼 게 있으면 아래를 False로
AUTO_OFF = True
if AUTO_OFF:
    push_results()
    from google.colab import runtime
    runtime.unassign()